# Preprocessing & Feature Engineering
## Credit Card Fraud Detection — Feature Pipeline

This notebook constructs **all 21 features** that will be used across training and inference.
Every transformation that learns from data (peer-group means, group boundaries) is **fitted
exclusively on the training set** and then applied to the test set, preventing any information
leakage across the chronological boundary (Day 139 → Day 140).

### Chronological Split
| Split | Days | Role |
|-------|------|------|
| Train | 0 – 139 | Model fitting, statistic computation |
| Test  | 140 – 182 | Evaluation only — no statistics derived from this window |

### Feature Catalogue
| # | Feature | Type | Target Scenario |
|---|---------|------|----------------|
| 1 | `TX_AMOUNT` | Numeric | Scenario 1 (Large Amount) |
| 2 | `hour` | Categorical Int8 | Temporal Risk |
| 3 | `distance` | Numeric | Spatial Constraints |
| 4 | `Z_score` | Numeric | Scenarios 1 & 3 |
| 5 | `amount_to_mean_ratio` | Numeric | Scenario 3 (5× Takeover) |
| 6 | `peer_group_amount_ratio` | Numeric | Peer Group Deviation |
| 7 | `is_night` | Binary | Nighttime Takeovers |
| 8 | `tx_count_1h` | Numeric | Bot Sprees / Card Testing |
| 9 | `tx_count_4h` | Numeric | Velocity Control |
| 10 | `night_velocity` | Numeric | Night Account Takeovers |
| 11 | `terminal_fraud_rate_3d` | Numeric | Scenario 2 (Skimming) |
| 12 | `terminal_fraud_rate_7d` | Numeric | Scenario 2 (Skimming) |
| 13 | `terminal_fraud_rate_28d` | Numeric | Scenario 2 (Skimming) |
| 14 | `night_fraud_rate` | Numeric | Night Skimming |
| 15 | `PREV_TX_AMOUNT_lag1` | Numeric | Sequence Baseline |
| 16 | `PREV_TX_AMOUNT_lag2` | Numeric | Sequence Baseline |
| 17 | `PREV_TX_AMOUNT_lag3` | Numeric | Sequence Baseline |
| 18 | `ratio_to_lag1` | Numeric | Scenario 3 (5× Takeover) |
| 19 | `ratio_to_lag2` | Numeric | Scenario 3 (Interleaved) |
| 20 | `ratio_to_lag3` | Numeric | Scenario 3 (Interleaved) |
| 21 | `is_test_tx_sequence` | Binary | Credential Testing Sequence |


## 1. Imports & Configuration

In [9]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from scipy.spatial import KDTree
from scipy import sparse
import scipy.sparse as sp

warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR    = Path("../data")
OUTPUT_DIR  = Path("../data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Constants ──────────────────────────────────────────────────────────────────
TRAIN_DAYS  = 140          # Days 0-139 are training
NIGHT_HOURS = {0, 1, 2, 3, 4}   # Hours flagged as high-risk nighttime
SMALL_TX    = 10.0         # Threshold for "test charge" detection
LARGE_TX    = 150.0        # Threshold for "credential abuse" detection
EPS         = 1e-9         # Division guard

print("Configuration loaded.")
print(f"  Training window : Days 0 – {TRAIN_DAYS - 1}")
print(f"  Test window     : Days {TRAIN_DAYS} – 182")
print(f"  Night hours     : {sorted(NIGHT_HOURS)}")


Configuration loaded.
  Training window : Days 0 – 139
  Test window     : Days 140 – 182
  Night hours     : [0, 1, 2, 3, 4]


## 2. Load Raw Data & Merge

Three source files are joined to produce the base dataframe:
- `customer_profiles.csv` — static customer spending profile (`mean_amount`, `std_amount`, coordinates)
- `terminal_profiles.csv` — static terminal coordinates
- `synthetic_fraud_transactions.csv` — transaction log with ground-truth fraud labels


In [10]:
customer_df  = pd.read_csv(DATA_DIR / "customer_profiles.csv")
terminal_df  = pd.read_csv(DATA_DIR / "terminal_profiles.csv")
synth_df     = pd.read_csv(DATA_DIR / "synthetic_fraud_transactions.csv")

# Parse datetime immediately so every downstream step has a consistent dtype
synth_df["TX_DATETIME"] = pd.to_datetime(synth_df["TX_DATETIME"])

# Merge: transactions ← customer profiles ← terminal profiles
df = (
    synth_df
    .merge(customer_df, on="CUSTOMER_ID", how="left")
    .merge(terminal_df, on="TERMINAL_ID", how="left")
)

print(f"Merged dataset shape : {df.shape}")
print(f"Columns              : {df.columns.tolist()}")
print(f"Date range           : {df['TX_DATETIME'].min()}  →  {df['TX_DATETIME'].max()}")
df.head(3)


Merged dataset shape : (1754155, 17)
Columns              : ['TRANSACTION_ID', 'TX_DATETIME', 'CUSTOMER_ID', 'TERMINAL_ID', 'TX_AMOUNT', 'TX_TIME_SECONDS', 'TX_TIME_DAYS', 'TX_FRAUD', 'TX_FRAUD_SCENARIO', 'x_customer_id', 'y_customer_id', 'mean_amount', 'std_amount', 'mean_nb_tx_per_day', 'nb_terminals', 'x_terminal_id', 'y_terminal_id']
Date range           : 2018-04-01 00:00:31  →  2018-09-30 23:59:57


,TRANSACTION_ID,TX_DATETIME,CUSTOMER_ID,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO,x_customer_id,y_customer_id,mean_amount,std_amount,mean_nb_tx_per_day,nb_terminals,x_terminal_id,y_terminal_id
0,0,2018-04-01 00:00:31,596,3156,57.16,31,0,0,0,14.186726,36.180124,41.255735,20.627868,3.789234,74,16.129403,35.482430
1,1,2018-04-01 00:02:10,4961,3412,81.51,130,0,0,0,70.297680,31.886146,85.494893,42.747447,3.946423,88,68.114957,34.904664
2,2,2018-04-01 00:07:56,2,1365,146.00,476,0,0,0,96.366276,38.344152,80.213879,40.106939,2.115580,70,92.927318,40.207817


## 3. Day Index

`TX_DAY` is an integer counting days from the first transaction in the dataset.
It is used as the join key when attaching rolling terminal fraud rates.


In [11]:
T0 = df["TX_DATETIME"].min().normalize()   # midnight of the first day
df["TX_DAY"] = (df["TX_DATETIME"] - T0).dt.days

print(f"TX_DAY range : {df['TX_DAY'].min()} – {df['TX_DAY'].max()}")
print(f"Train / test boundary at TX_DAY = {TRAIN_DAYS}")


TX_DAY range : 0 – 182
Train / test boundary at TX_DAY = 140


## 4. Temporal Features: `hour`, `is_night`, and `day_of_week`

| Feature | Formula | Notes |
|---------|---------|-------|
| `hour` | `TX_DATETIME.dt.hour` | 0-23; stored as `int8` |
| `is_night` | `1 if hour ∈ {0,1,2,3,4} else 0` | EDA showed elevated fraud rates in early-morning hours |
| `day_of_week` | `TX_DATETIME.dt.dayofweek` | 0=Monday … 6=Sunday; stored as `int8` |


In [12]:
df["hour"]     = df["TX_DATETIME"].dt.hour.astype("int8")
df["is_night"] = df["hour"].isin(NIGHT_HOURS).astype("int8")

print("Hour distribution (top 5):")
print(df["hour"].value_counts().head())
print(f"\nNight transactions : {df['is_night'].sum():,} / {len(df):,} ({df['is_night'].mean():.1%})")

df["day_of_week"] = df["TX_DATETIME"].dt.dayofweek.astype("int8")

print("\nday_of_week distribution (0=Mon … 6=Sun):")
print(df["day_of_week"].value_counts().sort_index())


Hour distribution (top 5):
hour
12    129108
11    128909
10    125804
13    125516
14    116965
Name: count, dtype: int64

Night transactions : 159,921 / 1,754,155 (9.1%)

day_of_week distribution (0=Mon … 6=Sun):
day_of_week
0    249707
1    248971
2    248716
3    249579
4    248797
5    248884
6    259501
Name: count, dtype: int64


## 5. Spatial Feature: `distance`

Euclidean distance between the customer's registered home coordinates and the terminal
where the transaction occurred.

$$\text{distance} = \sqrt{(x_{\text{customer}} - x_{\text{terminal}})^2 + (y_{\text{customer}} - y_{\text{terminal}})^2}$$

> **No leakage risk** — both coordinate sets come from static profiles, not from transactions.


In [13]:
df["distance"] = np.sqrt(
    (df["x_customer_id"] - df["x_terminal_id"]) ** 2 +
    (df["y_customer_id"] - df["y_terminal_id"]) ** 2
).astype("float32")

print(f"distance  →  min={df['distance'].min():.4f}  mean={df['distance'].mean():.4f}  max={df['distance'].max():.4f}")


distance  →  min=0.0116  mean=3.3090  max=5.0000


## 6. Amount Deviation Features: `Z_score` and `amount_to_mean_ratio`

Both features compare the current transaction amount against the customer's **historical
spending profile** (`mean_amount`, `std_amount`) from `customer_profiles.csv`.
Because these profile statistics are pre-computed from prior history (not from the synthetic
transaction log), there is no leakage.

| Feature | Formula |
|---------|---------|
| `Z_score` | $(\text{TX\_AMOUNT} - \text{mean\_amount}) / (\text{std\_amount} + \varepsilon)$ |
| `amount_to_mean_ratio` | $\text{TX\_AMOUNT} / (\text{mean\_amount} + \varepsilon)$ |


In [14]:
df["Z_score"] = (
    (df["TX_AMOUNT"] - df["mean_amount"]) / (df["std_amount"] + EPS)
).astype("float32")

df["amount_to_mean_ratio"] = (
    df["TX_AMOUNT"] / (df["mean_amount"] + EPS)
).astype("float32")

print("Z_score statistics:")
print(df["Z_score"].describe())
print("\namount_to_mean_ratio statistics:")
print(df["amount_to_mean_ratio"].describe())


Z_score statistics:
count    1.754155e+06
mean     7.605734e-02
std      1.068755e+00
min     -2.000000e+00
25%     -6.257755e-01
50%      3.028782e-02
75%      7.040875e-01
max      7.992155e+01
Name: Z_score, dtype: float64

amount_to_mean_ratio statistics:
count    1.754155e+06
mean     1.038029e+00
std      5.343777e-01
min      0.000000e+00
25%      6.871122e-01
50%      1.015144e+00
75%      1.352044e+00
max      4.096077e+01
Name: amount_to_mean_ratio, dtype: float64


## 6b. Customer Profile Features: `mean_nb_tx_per_day` and `nb_terminals`

Both columns live in `customer_profiles.csv` and are already present in `df`
after the merge in Section 2. No further computation is required.

| Feature | Description |
|---------|-------------|
| `mean_nb_tx_per_day` | Average number of transactions the customer makes per day. High-frequency customers have a denser footprint; an anomalous velocity spike stands out more sharply against a low baseline. |
| `nb_terminals` | Number of distinct terminals the customer has historically used. A low value means the customer rarely changes location — a transaction at an unfamiliar terminal is a stronger signal. |

> **No leakage risk** — both values are pre-computed from the customer's prior history
> and are static across the entire transaction log.


In [15]:
# ── Validate profile features are present and non-null after merge ────────────
profile_feat_cols = ["mean_nb_tx_per_day", "nb_terminals"]

print("Profile feature validation:")
print(f"  {'Feature':<25}  {'Dtype':>10}  {'Nulls':>8}  {'Min':>8}  {'Max':>8}")
print("  " + "-" * 65)
for col in profile_feat_cols:
    print(f"  {col:<25}  {str(df[col].dtype):>10}  "
          f"{df[col].isna().sum():>8,}  "
          f"{df[col].min():>8.2f}  "
          f"{df[col].max():>8.2f}")

train_mask_preview = df["TX_DAY"] < TRAIN_DAYS
print("\nmean_nb_tx_per_day statistics (train):")
print(df.loc[train_mask_preview, "mean_nb_tx_per_day"].describe())
print("\nnb_terminals statistics (train):")
print(df.loc[train_mask_preview, "nb_terminals"].describe())


Profile feature validation:
  Feature                         Dtype     Nulls       Min       Max
  -----------------------------------------------------------------


  mean_nb_tx_per_day            float64         0      0.00      4.00
  nb_terminals                    int64         0     22.00    106.00

mean_nb_tx_per_day statistics (train):
count    1.341934e+06
mean     2.651085e+00
std      9.522042e-01
min      9.223413e-04
25%      1.963000e+00
50%      2.836420e+00
75%      3.453090e+00
max      3.999912e+00
Name: mean_nb_tx_per_day, dtype: float64

nb_terminals statistics (train):
count    1.341934e+06
mean     7.531968e+01
std      1.238641e+01
min      2.200000e+01
25%      6.900000e+01
50%      7.700000e+01
75%      8.300000e+01
max      1.060000e+02
Name: nb_terminals, dtype: float64


## 7. Peer Group Feature: `peer_group_amount_ratio`

**Concept:** Customers are grouped into four spending tiers (quartiles) based on their
historical average spend (`mean_amount` from profiles). The ratio of the current transaction
to the peer group's average flags deviations relative to similar spenders.

**Leakage-safe pipeline:**
1. Assign every customer to a spending tier using `mean_amount` quartile thresholds — computed
   once on all customer profiles (static data, no transaction labels involved).
2. Compute `peer_mean_amount` (average TX_AMOUNT per tier) **only on training transactions**.
3. Map this lookup back to **all** transactions.


In [16]:
# Step 1 — Assign spending tier from customer profile (static, no leakage)
tier_bins   = customer_df["mean_amount"].quantile([0, 0.25, 0.50, 0.75, 1.0]).values
tier_labels = ["Q1_low", "Q2_mid_low", "Q3_mid_high", "Q4_high"]

customer_df["spending_tier"] = pd.cut(
    customer_df["mean_amount"],
    bins=tier_bins,
    labels=tier_labels,
    include_lowest=True
)

df = df.merge(
    customer_df[["CUSTOMER_ID", "spending_tier"]],
    on="CUSTOMER_ID",
    how="left"
)

print("Customers per spending tier:")
print(df.drop_duplicates("CUSTOMER_ID")["spending_tier"].value_counts().sort_index())


Customers per spending tier:
spending_tier
Q1_low         1248
Q2_mid_low     1249
Q3_mid_high    1245
Q4_high        1248
Name: count, dtype: int64


In [17]:
# Step 2 — Fit peer mean on TRAINING transactions only
train_mask       = df["TX_DAY"] < TRAIN_DAYS
peer_mean_lookup = (
    df[train_mask]
    .groupby("spending_tier")["TX_AMOUNT"]
    .mean()
    .rename("peer_mean_amount")
)

print("Peer mean TX_AMOUNT by spending tier (fitted on train only):")
print(peer_mean_lookup)


Peer mean TX_AMOUNT by spending tier (fitted on train only):
spending_tier
Q1_low         16.377857
Q2_mid_low     41.388603
Q3_mid_high    66.199135
Q4_high        90.963686
Name: peer_mean_amount, dtype: float64


In [18]:
# Step 3 — Apply to all transactions (train + test)
df = df.merge(peer_mean_lookup, on="spending_tier", how="left")

df["peer_group_amount_ratio"] = (
    df["TX_AMOUNT"] / (df["peer_mean_amount"] + EPS)
).astype("float32")

print(f"peer_group_amount_ratio  →  mean={df['peer_group_amount_ratio'].mean():.4f}  "
      f"max={df['peer_group_amount_ratio'].max():.2f}")
print(f"Null count: {df['peer_group_amount_ratio'].isna().sum()}")


peer_group_amount_ratio  →  mean=1.0000  max=35.87
Null count: 0


## 8. Customer Velocity Features: `tx_count_1h` and `tx_count_4h`

For each transaction, count how many transactions the **same customer** made in the
preceding 1 hour (or 4 hours), **excluding the current transaction itself** (`closed='left'`).

> **Leakage note:** `closed='left'` means the rolling window is `[T − window, T)` — the
> current event is never included. Sorting must be done by `(CUSTOMER_ID, TX_DATETIME)` first.


In [19]:
# Sort for time-ordered groupby operations — required for all sequential features
df = df.sort_values(["CUSTOMER_ID", "TX_DATETIME"]).reset_index(drop=True)

# Use TX_DATETIME as the index for time-based rolling
df_t = df.set_index("TX_DATETIME")

df_t["tx_count_1h"] = (
    df_t.groupby("CUSTOMER_ID")["TX_AMOUNT"]
    .transform(lambda x: x.rolling("1h", closed="left").count())
).fillna(0).astype("int16")

df_t["tx_count_4h"] = (
    df_t.groupby("CUSTOMER_ID")["TX_AMOUNT"]
    .transform(lambda x: x.rolling("4h", closed="left").count())
).fillna(0).astype("int16")

df = df_t.reset_index()  # restore TX_DATETIME as a column

print("tx_count_1h distribution:")
print(df["tx_count_1h"].value_counts().sort_index().head(10))
print(f"\ntx_count_4h  →  max={df['tx_count_4h'].max()}")


tx_count_1h distribution:
tx_count_1h
0    1531450
1     203811
2      17629
3       1180
4         81
5          4
Name: count, dtype: int64

tx_count_4h  →  max=8


## 9. Interaction Feature: `night_velocity`

Combines time-of-day risk and transaction velocity into a single interaction term:

$$\text{night\_velocity} = \text{is\_night} \times \text{tx\_count\_1h}$$

A value > 0 only fires during the high-risk nighttime window **and** when the customer
is transacting rapidly, targeting account takeover sprees at night.


In [20]:
df["night_velocity"] = (df["is_night"] * df["tx_count_1h"]).astype("int16")

print(f"night_velocity > 0 in {(df['night_velocity'] > 0).sum():,} transactions")
print(df["night_velocity"].value_counts().sort_index().head(10))


night_velocity > 0 in 7,282 transactions
night_velocity
0    1746873
1       7065
2        215
3          2
Name: count, dtype: int64


## 10. Terminal Rolling Fraud Rates: `terminal_fraud_rate_3d / 7d / 28d`

**Design:** For each terminal on each day `d`, compute the rolling average fraud rate over
the preceding 3, 7, or 28 days — looking at history from `[d − window, d)` (1-day shift
to prevent leakage of same-day labels).

**Pipeline:**
1. Aggregate daily fraud count and transaction count per terminal.
2. Build a full `(terminal × day)` grid so rolling is computed correctly even on quiet days.
3. Apply `shift(1)` before the rolling sum — the model never sees same-day fraud labels.
4. Merge rates back to the transaction level on `(TERMINAL_ID, TX_DAY)`.

> **Why this stops the "frozen index" problem:** A skimmer installed on Day 142 registers
> fraud on Day 143. By Day 144 the 3-day window `[141, 144)` already captures that spike,
> flagging Terminal A within 24-48 hours.

In [21]:
# ── Step 1: Daily terminal aggregation ────────────────────────────────────────
daily_terminal = (
    df.groupby(["TERMINAL_ID", "TX_DAY"])
    .agg(daily_fraud=("TX_FRAUD", "sum"), daily_count=("TX_FRAUD", "count"))
    .reset_index()
)

# ── Step 2: Full (terminal × day) grid so rolling never skips silent days ─────
all_terminals = df["TERMINAL_ID"].unique()
all_days      = range(int(df["TX_DAY"].min()), int(df["TX_DAY"].max()) + 1)

full_grid = pd.MultiIndex.from_product(
    [all_terminals, all_days], names=["TERMINAL_ID", "TX_DAY"]
).to_frame(index=False)

daily_terminal = (
    full_grid
    .merge(daily_terminal, on=["TERMINAL_ID", "TX_DAY"], how="left")
    .fillna(0)
    .sort_values(["TERMINAL_ID", "TX_DAY"])
    .reset_index(drop=True)
)

print(f"Daily terminal grid shape : {daily_terminal.shape}")


Daily terminal grid shape : (1830000, 4)


In [22]:
# ── Step 3: Rolling rates with 1-day shift ────────────────────────────────────
def rolling_fraud_rate(group: pd.DataFrame, window: int) -> pd.Series:
    """
    For a single terminal's daily time series, compute the rolling
    fraud rate over `window` days, shifted 1 day forward to prevent
    same-day label leakage.

    Formula:
        rate[d] = sum(fraud[d - window : d - 1]) / max(sum(count[d - window : d - 1]), 1)
    """
    fraud_roll = group["daily_fraud"].shift(1).rolling(window, min_periods=1).sum()
    count_roll = group["daily_count"].shift(1).rolling(window, min_periods=1).sum().clip(lower=1)
    return (fraud_roll / count_roll).fillna(0)

for window, col in [(3, "terminal_fraud_rate_3d"),
                    (7, "terminal_fraud_rate_7d"),
                    (28, "terminal_fraud_rate_28d")]:
    daily_terminal[col] = (
        daily_terminal
        .groupby("TERMINAL_ID", group_keys=False)
        .apply(lambda g: rolling_fraud_rate(g, window))
    ).astype("float32")

print("Rolling rates computed. Sample:")
print(daily_terminal[["TERMINAL_ID", "TX_DAY",
                       "terminal_fraud_rate_3d",
                       "terminal_fraud_rate_7d",
                       "terminal_fraud_rate_28d"]].head(10))


Rolling rates computed. Sample:
   TERMINAL_ID  TX_DAY  terminal_fraud_rate_3d  terminal_fraud_rate_7d  \
0            0       0                     0.0                     0.0   
1            0       1                     0.0                     0.0   
2            0       2                     0.0                     0.0   
3            0       3                     0.0                     0.0   
4            0       4                     0.0                     0.0   
5            0       5                     0.0                     0.0   
6            0       6                     0.0                     0.0   
7            0       7                     0.0                     0.0   
8            0       8                     0.0                     0.0   
9            0       9                     0.0                     0.0   

   terminal_fraud_rate_28d  
0                      0.0  
1                      0.0  
2                      0.0  
3                      0.0  
4       

In [23]:
# ── Step 4: Merge back to transaction level ───────────────────────────────────
rate_cols = ["TERMINAL_ID", "TX_DAY",
             "terminal_fraud_rate_3d", "terminal_fraud_rate_7d", "terminal_fraud_rate_28d"]

df = df.merge(daily_terminal[rate_cols], on=["TERMINAL_ID", "TX_DAY"], how="left")

for col in ["terminal_fraud_rate_3d", "terminal_fraud_rate_7d", "terminal_fraud_rate_28d"]:
    df[col] = df[col].fillna(0).astype("float32")

print("Terminal fraud rates attached. Null check:")
print(df[["terminal_fraud_rate_3d",
          "terminal_fraud_rate_7d",
          "terminal_fraud_rate_28d"]].isna().sum())


Terminal fraud rates attached. Null check:
terminal_fraud_rate_3d     0
terminal_fraud_rate_7d     0
terminal_fraud_rate_28d    0
dtype: int64


## 11. Night Fraud Rate: `night_fraud_rate`

Same rolling window logic as terminal rates (7-day, 1-day shift), but computed
**only from nighttime transactions** (`is_night == 1`).

This surfaces terminals that are disproportionately targeted by nighttime fraud
even when their overall daytime fraud rate looks benign.


In [24]:
# Aggregate nighttime transactions per terminal per day
daily_night = (
    df[df["is_night"] == 1]
    .groupby(["TERMINAL_ID", "TX_DAY"])
    .agg(nightly_fraud=("TX_FRAUD", "sum"), nightly_count=("TX_FRAUD", "count"))
    .reset_index()
)

daily_night = (
    full_grid
    .merge(daily_night, on=["TERMINAL_ID", "TX_DAY"], how="left")
    .fillna(0)
    .sort_values(["TERMINAL_ID", "TX_DAY"])
    .reset_index(drop=True)
    .rename(columns={"nightly_fraud": "daily_fraud", "nightly_count": "daily_count"})
)

daily_night["night_fraud_rate"] = (
    daily_night
    .groupby("TERMINAL_ID", group_keys=False)
    .apply(lambda g: rolling_fraud_rate(g, 7))   # 7-day window, same as terminal_fraud_rate_7d
).astype("float32")

df = df.merge(
    daily_night[["TERMINAL_ID", "TX_DAY", "night_fraud_rate"]],
    on=["TERMINAL_ID", "TX_DAY"],
    how="left"
)
df["night_fraud_rate"] = df["night_fraud_rate"].fillna(0).astype("float32")

print(f"night_fraud_rate  →  mean={df['night_fraud_rate'].mean():.6f}  "
      f"max={df['night_fraud_rate'].max():.4f}")
print(f"Null count: {df['night_fraud_rate'].isna().sum()}")


night_fraud_rate  →  mean=0.003877  max=1.0000
Null count: 0


In [25]:
df.columns

Index(['TX_DATETIME', 'TRANSACTION_ID', 'CUSTOMER_ID', 'TERMINAL_ID',
       'TX_AMOUNT', 'TX_TIME_SECONDS', 'TX_TIME_DAYS', 'TX_FRAUD',
       'TX_FRAUD_SCENARIO', 'x_customer_id', 'y_customer_id', 'mean_amount',
       'std_amount', 'mean_nb_tx_per_day', 'nb_terminals', 'x_terminal_id',
       'y_terminal_id', 'TX_DAY', 'hour', 'is_night', 'day_of_week',
       'distance', 'Z_score', 'amount_to_mean_ratio', 'spending_tier',
       'peer_mean_amount', 'peer_group_amount_ratio', 'tx_count_1h',
       'tx_count_4h', 'night_velocity', 'terminal_fraud_rate_3d',
       'terminal_fraud_rate_7d', 'terminal_fraud_rate_28d',
       'night_fraud_rate'],
      dtype='str')

In [26]:
# ============================================================
# 12. Spatial Neighborhood Fraud Rate (7-day)
# ============================================================

print("Calculating neigh_fraud_rate...")
daily_stats = (
    df[
        ["TERMINAL_ID", "TX_DAY", "terminal_fraud_rate_7d"]
    ]
    .drop_duplicates(["TERMINAL_ID", "TX_DAY"])
    .sort_values(["TERMINAL_ID", "TX_DAY"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Build neighborhood graph using terminal coordinates
# ------------------------------------------------------------
terminal_coords = (
    df[["TERMINAL_ID", "x_terminal_id", "y_terminal_id"]]
    .drop_duplicates("TERMINAL_ID")
    .sort_values("TERMINAL_ID")
)

coords = terminal_coords[["x_terminal_id", "y_terminal_id"]].values

tree = KDTree(coords)

# radius can be tuned (1.0 was used in the original implementation)
neighbors = tree.query_ball_point(coords, r=1.0)

n_terminals = len(terminal_coords)

rows, cols, weights = [], [], []

for i, neighs in enumerate(neighbors):
    if len(neighs) == 0:
        continue

    w = 1.0 / len(neighs)

    for j in neighs:
        rows.append(i)
        cols.append(j)
        weights.append(w)

W = sparse.csr_matrix(
    (weights, (rows, cols)),
    shape=(n_terminals, n_terminals)
)

all_terminals = np.sort(df["TERMINAL_ID"].unique())
all_days = np.sort(df["TX_DAY"].unique())

full_grid = (
    pd.MultiIndex.from_product(
        [all_terminals, all_days],
        names=["TERMINAL_ID", "TX_DAY"]
    )
    .to_frame(index=False)
)

# ------------------------------------------------------------
# Build (terminal × day) fraud-rate matrix
# ------------------------------------------------------------
grid = (
    full_grid.merge(
        daily_stats[
            ["TERMINAL_ID", "TX_DAY", "terminal_fraud_rate_7d"]
        ],
        on=["TERMINAL_ID", "TX_DAY"],
        how="left"
    )
    .fillna({"terminal_fraud_rate_7d": 0.0})
)

F = (
    grid.pivot(
        index="TX_DAY",
        columns="TERMINAL_ID",
        values="terminal_fraud_rate_7d"
    )
    .sort_index(axis=1)
    .values
)

# ------------------------------------------------------------
# Compute neighborhood fraud rate
# ------------------------------------------------------------
neighbor_rates = F @ W.T

grid["neigh_fraud_rate"] = neighbor_rates.flatten().astype(np.float32)

# ------------------------------------------------------------
# Merge back to transaction level
# ------------------------------------------------------------
df = df.merge(
    grid[
        ["TERMINAL_ID", "TX_DAY", "neigh_fraud_rate"]
    ],
    on=["TERMINAL_ID", "TX_DAY"],
    how="left"
)

df["neigh_fraud_rate"] = (
    df["neigh_fraud_rate"]
    .fillna(0)
    .astype(np.float32)
)

print(
    f"neigh_fraud_rate → mean={df['neigh_fraud_rate'].mean():.6f}, "
    f"max={df['neigh_fraud_rate'].max():.4f}"
)
print(f"Null count: {df['neigh_fraud_rate'].isna().sum()}")

Calculating neigh_fraud_rate...
neigh_fraud_rate → mean=0.004922, max=1.0000
Null count: 0


## 12. Customer Amount Lag Features: `PREV_TX_AMOUNT_lag1/2/3`

For each transaction, look back at the customer's **1st, 2nd, and 3rd previous**
transaction amounts (ordered by `TX_DATETIME`).

> **Important:** The sort on `(CUSTOMER_ID, TX_DATETIME)` was already applied in
> Section 8. We re-sort here for clarity before computing lags.
>
> Null values (the customer's first 1-3 transactions have no history) are filled
> with the customer's `mean_amount` from their profile, preserving the
> ratio features' scale without introducing extreme values.


In [27]:
# Ensure chronological order per customer
df = df.sort_values(["CUSTOMER_ID", "TX_DATETIME"]).reset_index(drop=True)

# Shift by 1, 2, 3 within each customer group
df["PREV_TX_AMOUNT_lag1"] = df.groupby("CUSTOMER_ID")["TX_AMOUNT"].shift(1)
df["PREV_TX_AMOUNT_lag2"] = df.groupby("CUSTOMER_ID")["TX_AMOUNT"].shift(2)
df["PREV_TX_AMOUNT_lag3"] = df.groupby("CUSTOMER_ID")["TX_AMOUNT"].shift(3)

# Fill NaN at start of each customer's history with customer mean (from profile)
for lag_col in ["PREV_TX_AMOUNT_lag1", "PREV_TX_AMOUNT_lag2", "PREV_TX_AMOUNT_lag3"]:
    null_mask        = df[lag_col].isna()
    df.loc[null_mask, lag_col] = df.loc[null_mask, "mean_amount"]

df[["PREV_TX_AMOUNT_lag1",
    "PREV_TX_AMOUNT_lag2",
    "PREV_TX_AMOUNT_lag3"]] = df[["PREV_TX_AMOUNT_lag1",
                                   "PREV_TX_AMOUNT_lag2",
                                   "PREV_TX_AMOUNT_lag3"]].astype("float32")

print("Lag feature null counts (should be 0 after fill):")
print(df[["PREV_TX_AMOUNT_lag1","PREV_TX_AMOUNT_lag2","PREV_TX_AMOUNT_lag3"]].isna().sum())
print("\nSample:")
print(df[["CUSTOMER_ID","TX_DATETIME","TX_AMOUNT",
          "PREV_TX_AMOUNT_lag1","PREV_TX_AMOUNT_lag2","PREV_TX_AMOUNT_lag3"]].head(8))


Lag feature null counts (should be 0 after fill):
PREV_TX_AMOUNT_lag1    0
PREV_TX_AMOUNT_lag2    0
PREV_TX_AMOUNT_lag3    0
dtype: int64

Sample:
   CUSTOMER_ID         TX_DATETIME  TX_AMOUNT  PREV_TX_AMOUNT_lag1  \
0            0 2018-04-01 07:19:05     123.59            62.262520   
1            0 2018-04-01 18:00:16      77.34           123.589996   
2            0 2018-04-01 19:02:02      46.51            77.339996   
3            0 2018-04-02 08:51:06      54.72            46.509998   
4            0 2018-04-02 14:05:38      63.30            54.720001   
5            0 2018-04-02 15:13:02      32.35            63.299999   
6            0 2018-04-02 15:46:51      13.59            32.349998   
7            0 2018-04-02 20:24:47      51.89            13.590000   

   PREV_TX_AMOUNT_lag2  PREV_TX_AMOUNT_lag3  
0            62.262520            62.262520  
1            62.262520            62.262520  
2           123.589996            62.262520  
3            77.339996           123.5

## 13. Sequence Ratio Features: `ratio_to_lag1/2/3`

$$\text{ratio\_to\_lagN} = \frac{\text{TX\_AMOUNT}}{\text{PREV\_TX\_AMOUNT\_lagN} + \varepsilon}$$

These detect the **5× signature** of Scenario 3 (credential takeovers) even when normal
transactions are interleaved between fraudulent ones.

| Ratio | What it catches |
|-------|----------------|
| `ratio_to_lag1` | Direct surge vs last tx |
| `ratio_to_lag2` | Surge when 1 normal tx is interleaved |
| `ratio_to_lag3` | Surge when 2 normal txs are interleaved |


In [28]:
df["ratio_to_lag1"] = (df["TX_AMOUNT"] / (df["PREV_TX_AMOUNT_lag1"] + EPS)).astype("float32")
df["ratio_to_lag2"] = (df["TX_AMOUNT"] / (df["PREV_TX_AMOUNT_lag2"] + EPS)).astype("float32")
df["ratio_to_lag3"] = (df["TX_AMOUNT"] / (df["PREV_TX_AMOUNT_lag3"] + EPS)).astype("float32")

print("Ratio feature statistics:")
print(df[["ratio_to_lag1","ratio_to_lag2","ratio_to_lag3"]].describe())


Ratio feature statistics:
       ratio_to_lag1  ratio_to_lag2  ratio_to_lag3
count   1.754155e+06   1.754155e+06   1.754155e+06
mean    6.979964e+05   6.532569e+05   6.873588e+05
std     1.896407e+08   1.964127e+08   1.897981e+08
min     0.000000e+00   0.000000e+00   0.000000e+00
25%     6.200622e-01   6.201316e-01   6.199931e-01
50%     1.000945e+00   9.999999e-01   1.000809e+00
75%     1.613065e+00   1.610999e+00   1.610294e+00
max     9.266000e+10   1.079700e+11   1.061200e+11


## 14. Credential Testing Sequence Flag: `is_test_tx_sequence`

Fraudsters commonly make a **small test charge** (< \$10) to verify a stolen card
is active before immediately executing a **large takeover purchase** (> \$150).

$$\text{is\_test\_tx\_sequence} = \mathbb{1}[\text{PREV\_TX\_AMOUNT\_lag1} < 10 \;\land\; \text{TX\_AMOUNT} > 150]$$


In [29]:
df["is_test_tx_sequence"] = (
    (df["PREV_TX_AMOUNT_lag1"] < SMALL_TX) &
    (df["TX_AMOUNT"] > LARGE_TX)
).astype("int8")

flag_count = df["is_test_tx_sequence"].sum()
fraud_in_flag = df.loc[df["is_test_tx_sequence"] == 1, "TX_FRAUD"].mean()

print(f"Transactions flagged : {flag_count:,}  ({flag_count/len(df):.2%} of all txs)")
print(f"Fraud rate in flagged: {fraud_in_flag:.2%}")


Transactions flagged : 737  (0.04% of all txs)
Fraud rate in flagged: 12.08%


## 15. Chronological Train / Test Split

All features are now fully computed. We split on `TX_DAY` using the pre-defined
boundary (Day 140). No shuffling — the temporal split preserves the realistic
scenario where the model is trained on historical data and evaluated on future data.


In [34]:
# ── Final feature list ────────────────────────────────────────────────────────
FEATURE_COLS = [
    "TX_AMOUNT",
    "hour",
    "distance",
    "Z_score",
    "amount_to_mean_ratio",
    "peer_group_amount_ratio",
    "is_night",
    "tx_count_1h",
    "tx_count_4h",
    "night_velocity",
    "terminal_fraud_rate_3d",
    "terminal_fraud_rate_7d",
    "terminal_fraud_rate_28d",
    "night_fraud_rate",
    "PREV_TX_AMOUNT_lag1",
    "PREV_TX_AMOUNT_lag2",
    "PREV_TX_AMOUNT_lag3",
    "ratio_to_lag1",
    "ratio_to_lag2",
    "ratio_to_lag3",
    "is_test_tx_sequence",
    "neigh_fraud_rate",
    # ── Customer profile features (from customer_profiles.csv) ──────────
    "mean_nb_tx_per_day",
    "nb_terminals",
    "day_of_week",
]

TARGET_COL = "TX_FRAUD"

METADATA_COLS = [
    "TRANSACTION_ID", "CUSTOMER_ID", "TERMINAL_ID",
    "TX_DATETIME", "TX_DAY", "TX_FRAUD_SCENARIO",
]

ALL_KEEP = METADATA_COLS + FEATURE_COLS + [TARGET_COL]

# ── Split ─────────────────────────────────────────────────────────────────────
train_df = df[df["TX_DAY"] < TRAIN_DAYS][ALL_KEEP].copy()
test_df  = df[df["TX_DAY"] >= TRAIN_DAYS][ALL_KEEP].copy()

print(f"Train  →  {train_df.shape[0]:>10,} rows  |  Fraud rate: {train_df[TARGET_COL].mean():.4%}")
print(f"Test   →  {test_df.shape[0]:>10,} rows  |  Fraud rate: {test_df[TARGET_COL].mean():.4%}")


Train  →   1,341,934 rows  |  Fraud rate: 0.8203%
Test   →     412,221 rows  |  Fraud rate: 0.8910%


## 16. Feature Consistency Validation

Quick checks to catch common preprocessing errors before saving:
- No feature is entirely null
- No feature has zero variance in both splits (would be useless to a model)
- Feature count matches expectations


In [35]:
print("=" * 70)
print(f"{'Feature':<35}  {'Train Null%':>10}  {'Test Null%':>10}  {'Train Std':>10}")
print("=" * 70)

issues = []
for col in FEATURE_COLS:
    tr_null = train_df[col].isna().mean() * 100
    te_null = test_df[col].isna().mean() * 100
    tr_std  = train_df[col].std()
    flag    = "  ⚠" if (tr_null > 0 or te_null > 0 or tr_std == 0) else ""
    print(f"{col:<35}  {tr_null:>9.2f}%  {te_null:>9.2f}%  {tr_std:>10.4f}{flag}")
    if flag:
        issues.append(col)

print("=" * 70)
if issues:
    print(f"\n⚠ Features needing attention: {issues}")
else:
    print(f"\n✅ All {len(list(FEATURE_COLS))} features pass null and variance checks.")


Feature                              Train Null%  Test Null%   Train Std
TX_AMOUNT                                 0.00%       0.00%     42.1061
hour                                      0.00%       0.00%      5.0562
distance                                  0.00%       0.00%      1.1879
Z_score                                   0.00%       0.00%      1.0631
amount_to_mean_ratio                      0.00%       0.00%      0.5316
peer_group_amount_ratio                   0.00%       0.00%      0.5742
is_night                                  0.00%       0.00%      0.2876
tx_count_1h                               0.00%       0.00%      0.3792
tx_count_4h                               0.00%       0.00%      0.7824
night_velocity                            0.00%       0.00%      0.0670
terminal_fraud_rate_3d                    0.00%       0.00%      0.0764
terminal_fraud_rate_7d                    0.00%       0.00%      0.0712
terminal_fraud_rate_28d                   0.00%       0.00%    

In [36]:
print("\nTrain feature summary:")
train_df[FEATURE_COLS].describe().T[["mean","std","min","max"]]



Train feature summary:


,mean,std,min,max
TX_AMOUNT,53.627248,4.210607e+01,0.000000,1.108850e+03
hour,11.501746,5.056181e+00,0.000000,2.300000e+01
distance,3.308942,1.187903e+00,0.011552,4.999990e+00
Z_score,0.075728,1.063116e+00,-2.000000,4.833933e+01
amount_to_mean_ratio,1.037864,5.315579e-01,0.000000,2.516966e+01
peer_group_amount_ratio,1.000000,5.741585e-01,0.000000,2.495748e+01
is_night,0.090993,2.875997e-01,0.000000,1.000000e+00
tx_count_1h,0.138271,3.791841e-01,0.000000,5.000000e+00
tx_count_4h,0.532195,7.824003e-01,0.000000,8.000000e+00
night_velocity,0.004260,6.699281e-02,0.000000,3.000000e+00


## 17. Save Processed Datasets

| File | Contents |
|------|----------|
| `train_features.csv` | 21 engineered features + target + metadata (Days 0-139) |
| `test_features.csv` | Same schema applied to test window (Days 140-182) |
| `full_features.csv` | Both splits combined — useful for cross-validation or LSTM sequence prep |


In [37]:
train_path = OUTPUT_DIR / "train_features.csv"
test_path  = OUTPUT_DIR / "test_features.csv"
full_path  = OUTPUT_DIR / "full_features.csv"

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path,  index=False)
df[ALL_KEEP].to_csv(full_path, index=False)

print(f"Saved → {train_path}   ({train_df.shape[0]:,} rows)")
print(f"Saved → {test_path}    ({test_df.shape[0]:,} rows)")
print(f"Saved → {full_path}    ({len(df):,} rows)")


Saved → ..\data\train_features.csv   (1,341,934 rows)
Saved → ..\data\test_features.csv    (412,221 rows)
Saved → ..\data\full_features.csv    (1,754,155 rows)


## 18. Split the existing test set into Holdout and OOT

In [39]:
HOLDOUT_END_DAY = 162      # exclusive

holdout_df = test_df[test_df["TX_DAY"] < HOLDOUT_END_DAY].copy()

oot_df = test_df[test_df["TX_DAY"] >= HOLDOUT_END_DAY].copy()

print(f"Train    : {train_df['TX_DAY'].min()} - {train_df['TX_DAY'].max()}")
print(f"Holdout  : {holdout_df['TX_DAY'].min()} - {holdout_df['TX_DAY'].max()}")
print(f"OOT      : {oot_df['TX_DAY'].min()} - {oot_df['TX_DAY'].max()}")

Train    : 0 - 139
Holdout  : 140 - 161
OOT      : 162 - 182


In [44]:
train_path    = OUTPUT_DIR / "train_features.parquet"
holdout_path  = OUTPUT_DIR / "holdout_features.parquet"
oot_path      = OUTPUT_DIR / "oot_features.parquet"
full_path     = OUTPUT_DIR / "full_features.parquet"

train_df.to_parquet(train_path, engine="fastparquet", index=False)

holdout_df.to_parquet(holdout_path, engine="fastparquet", index=False)

oot_df.to_parquet(oot_path, engine="fastparquet", index=False)

df[ALL_KEEP].to_parquet(full_path, engine="fastparquet", index=False)

print("Saved:")
print(train_path)
print(holdout_path)
print(oot_path)
print(full_path)

Saved:
..\data\train_features.parquet
..\data\holdout_features.parquet
..\data\oot_features.parquet
..\data\full_features.parquet


In [38]:
scenario_map = {
    "TX_AMOUNT":                ("Numeric",   "Scenario 1 — Large Amount"),
    "hour":                     ("Cat Int8",  "Temporal Risk Profile"),
    "distance":                 ("Numeric",   "Spatial Constraints"),
    "Z_score":                  ("Numeric",   "Scenarios 1 & 3 — Outliers"),
    "amount_to_mean_ratio":     ("Numeric",   "Scenario 3 — 5× Takeover"),
    "peer_group_amount_ratio":  ("Numeric",   "Peer Group Deviation"),
    "is_night":                 ("Binary",    "Nighttime Takeovers"),
    "tx_count_1h":              ("Int16",     "Bot Sprees / Card Testing"),
    "tx_count_4h":              ("Int16",     "Velocity Control"),
    "night_velocity":           ("Int16",     "Night Account Takeovers"),

    "terminal_fraud_rate_3d":   ("Numeric",   "Scenario 2 — Skimming (fast)"),
    "terminal_fraud_rate_7d":   ("Numeric",   "Scenario 2 — Skimming (medium)"),
    "terminal_fraud_rate_28d":  ("Numeric",   "Scenario 2 — Skimming (long)"),
    "night_fraud_rate":         ("Numeric",   "Night Skimming"),
    "neigh_fraud_rate":         ("Numeric",   "Spatial Fraud Propagation"),

    "PREV_TX_AMOUNT_lag1":      ("Numeric",   "Sequence Baseline"),
    "PREV_TX_AMOUNT_lag2":      ("Numeric",   "Sequence Baseline"),
    "PREV_TX_AMOUNT_lag3":      ("Numeric",   "Sequence Baseline"),
    "ratio_to_lag1":            ("Numeric",   "Scenario 3 — Direct Surge"),
    "ratio_to_lag2":            ("Numeric",   "Scenario 3 — Interleaved Bypass"),
    "ratio_to_lag3":            ("Numeric",   "Scenario 3 — Interleaved Bypass"),
    "is_test_tx_sequence":      ("Binary",    "Credential Testing Sequence"),
    # ── Customer profile features ──────────────────────────────────────────
    "mean_nb_tx_per_day":       ("Numeric",   "Customer Activity Level"),
    "nb_terminals":             ("Numeric",   "Customer Mobility / Exposure"),
    # ── Additional temporal feature ────────────────────────────────────────
    "day_of_week":              ("Cat Int8",  "Weekly Behavioural Pattern"),
}


summary = pd.DataFrame(
    [(f, t, s) for f, (t, s) in scenario_map.items()],
    columns=["Feature", "Type", "Target Scenario"]
)

print("\n✅ Feature Engineering Complete — 24 features ready for modeling.")
summary


✅ Feature Engineering Complete — 24 features ready for modeling.


,Feature,Type,Target Scenario
0,TX_AMOUNT,Numeric,Scenario 1 — Large Amount
1,hour,Cat Int8,Temporal Risk Profile
2,distance,Numeric,Spatial Constraints
3,Z_score,Numeric,Scenarios 1 & 3 — Outliers
4,amount_to_mean_ratio,Numeric,Scenario 3 — 5× Takeover
5,peer_group_amount_ratio,Numeric,Peer Group Deviation
6,is_night,Binary,Nighttime Takeovers
7,tx_count_1h,Int16,Bot Sprees / Card Testing
8,tx_count_4h,Int16,Velocity Control
9,night_velocity,Int16,Night Account Takeovers
